# Confirmatory Human-Validation Materials

This notebook creates blinded human-validation materials for the 60-family confirmatory stimulus bank.

## Sentence-level validation design

The verified stimulus bank contains:

- 60 factual families
- 4 conditions per family
- 240 sentences in total

The four conditions are:

1. True + Plain
2. True + Formal
3. False + Plain
4. False + Formal

Four balanced validation lists will be created.

Each list will contain:

- exactly one sentence from each factual family;
- 60 sentences in total;
- 15 True + Plain sentences;
- 15 True + Formal sentences;
- 15 False + Plain sentences;
- 15 False + Formal sentences.

Across the four lists, every one of the 240 candidate sentences will appear exactly once.

Participants will not see family IDs, truth labels, register labels, domains, or experimental hypotheses.

In [3]:
import pandas as pd
import numpy as np

BANK_FILE = "/content/confirmatory_candidate_bank_F001_F060_verified.csv"

bank = pd.read_csv(BANK_FILE)

print("Rows:", len(bank))
print("Unique family IDs:", bank["family_id"].nunique())

display(bank.head())

FileNotFoundError: [Errno 2] No such file or directory: '/content/confirmatory_candidate_bank_F001_F060_verified.csv'

In [ ]:
conditions = [
    "true_plain",
    "true_formal",
    "false_plain",
    "false_formal"
]

list_names = [
    "List_A",
    "List_B",
    "List_C",
    "List_D"
]

assignment_rows = []

for family_index, row in bank.reset_index(drop=True).iterrows():

    for list_index, list_name in enumerate(list_names):

        condition = conditions[
            (family_index + list_index) % 4
        ]

        truth, register = condition.split("_")

        assignment_rows.append({
            "list_name": list_name,
            "family_id": row["family_id"],
            "domain": row["domain"],
            "relation": row["relation"],
            "condition": condition,
            "truth": truth,
            "register": register,
            "sentence": row[condition]
        })

sentence_master = pd.DataFrame(assignment_rows)

print("Total assignments:", len(sentence_master))

display(sentence_master.head(12))

In [ ]:
print("SENTENCES PER LIST")
print("------------------")

print(
    sentence_master["list_name"]
    .value_counts()
    .sort_index()
)

print("\nCONDITIONS PER LIST")
print("-------------------")

condition_check = pd.crosstab(
    sentence_master["list_name"],
    sentence_master["condition"]
)

display(condition_check)

In [ ]:
family_per_list_check = (
    sentence_master
    .groupby(["list_name", "family_id"])
    .size()
)

print(
    "Any list containing more than one sentence from the same family:",
    (family_per_list_check > 1).sum()
)

print(
    "Unique sentence assignments:",
    sentence_master[
        ["family_id", "condition"]
    ].drop_duplicates().shape[0]
)

In [ ]:
omission_groups = {
    "G1": [
        "F001", "F005", "F006", "F007", "F011",
        "F012", "F014", "F016", "F020", "F023",
        "F026", "F032", "F043", "F045", "F050"
    ],

    "G2": [
        "F002", "F013", "F015", "F017", "F027",
        "F028", "F029", "F036", "F037", "F042",
        "F046", "F048", "F051", "F055", "F060"
    ],

    "G3": [
        "F008", "F010", "F019", "F021", "F022",
        "F024", "F025", "F030", "F031", "F038",
        "F041", "F044", "F052", "F057", "F059"
    ],

    "G4": [
        "F003", "F004", "F009", "F018", "F033",
        "F034", "F035", "F039", "F040", "F047",
        "F049", "F053", "F054", "F056", "F058"
    ]
}

for group_name, family_ids in omission_groups.items():
    print(
        group_name,
        "families:",
        len(family_ids)
    )

all_omitted = [
    family_id
    for family_ids in omission_groups.values()
    for family_id in family_ids
]

print(
    "Total assigned families:",
    len(all_omitted)
)

print(
    "Unique assigned families:",
    len(set(all_omitted))
)

In [ ]:
condition_order = {
    "true_plain": 0,
    "true_formal": 1,
    "false_plain": 2,
    "false_formal": 3
}

sentence_master["_condition_order"] = (
    sentence_master["condition"]
    .map(condition_order)
)

sentence_master = (
    sentence_master
    .sort_values(
        [
            "family_id",
            "_condition_order"
        ]
    )
    .reset_index(drop=True)
)

sentence_master["stimulus_id"] = [
    f"S{i:03d}"
    for i in range(
        1,
        len(sentence_master) + 1
    )
]

sentence_master = sentence_master.drop(
    columns="_condition_order"
)

print(
    "Total stimulus IDs:",
    sentence_master["stimulus_id"].nunique()
)

display(
    sentence_master[
        [
            "stimulus_id",
            "family_id",
            "condition",
            "list_name",
            "sentence"
        ]
    ].head(8)
)

In [ ]:
variant_rows = []

for list_index, list_name in enumerate(
    ["List_A", "List_B", "List_C", "List_D"]
):

    base_list = sentence_master[
        sentence_master["list_name"] == list_name
    ].copy()

    for group_index, (
        group_name,
        omitted_families
    ) in enumerate(
        omission_groups.items(),
        start=1
    ):

        variant_name = (
            f"{list_name}_V{group_index}"
        )

        variant = base_list[
            ~base_list["family_id"].isin(
                omitted_families
            )
        ].copy()

        # Reproducible independent randomization
        random_seed = (
            2026
            + list_index * 100
            + group_index
        )

        variant = (
            variant
            .sample(
                frac=1,
                random_state=random_seed
            )
            .reset_index(drop=True)
        )

        variant["variant_name"] = variant_name
        variant["presentation_order"] = (
            range(
                1,
                len(variant) + 1
            )
        )

        # Three blocks of 15
        variant["block"] = (
            (
                variant["presentation_order"]
                - 1
            ) // 15
            + 1
        )

        variant_rows.append(
            variant
        )

validation_variants = pd.concat(
    variant_rows,
    ignore_index=True
)

print(
    "Total variant rows:",
    len(validation_variants)
)

print(
    "Number of variants:",
    validation_variants[
        "variant_name"
    ].nunique()
)

In [ ]:
print("SENTENCES PER FORM")
print("------------------")

print(
    validation_variants
    .groupby("variant_name")
    .size()
)

print("\nUNIQUE FAMILIES PER FORM")
print("------------------------")

print(
    validation_variants
    .groupby("variant_name")[
        "family_id"
    ]
    .nunique()
)

print("\nSTIMULUS COVERAGE")
print("-----------------")

stimulus_coverage = (
    validation_variants[
        "stimulus_id"
    ]
    .value_counts()
)

print(
    "Unique stimuli:",
    stimulus_coverage.size
)

print(
    "Minimum appearances:",
    stimulus_coverage.min()
)

print(
    "Maximum appearances:",
    stimulus_coverage.max()
)

print("\nCONDITION BALANCE")
print("-----------------")

condition_balance = pd.crosstab(
    validation_variants[
        "variant_name"
    ],
    validation_variants[
        "condition"
    ]
)

display(condition_balance)

In [ ]:
n_variants = (
    validation_variants[
        "variant_name"
    ].nunique()
)

usable_raters_per_variant = 5

total_usable_participants = (
    n_variants
    * usable_raters_per_variant
)

ratings_per_stimulus = (
    3
    * usable_raters_per_variant
)

print(
    "Validation forms:",
    n_variants
)

print(
    "Usable participants per form:",
    usable_raters_per_variant
)

print(
    "Total usable participants:",
    total_usable_participants
)

print(
    "Ratings per stimulus:",
    ratings_per_stimulus
)

In [ ]:
print(
    "Number of variants:",
    validation_variants["variant_name"].nunique()
)

stimulus_coverage = (
    validation_variants["stimulus_id"]
    .value_counts()
)

print(
    "Unique stimuli:",
    stimulus_coverage.size
)

print(
    "Minimum appearances:",
    stimulus_coverage.min()
)

print(
    "Maximum appearances:",
    stimulus_coverage.max()
)

print("\nCONDITION BALANCE")
print("-----------------")

condition_balance = pd.crosstab(
    validation_variants["variant_name"],
    validation_variants["condition"]
)

display(condition_balance)

In [ ]:
# Check that every form contains exactly 45 sentences
form_sizes = (
    validation_variants
    .groupby("variant_name")
    .size()
)

# Check that every form has 45 unique factual families
family_counts = (
    validation_variants
    .groupby("variant_name")["family_id"]
    .nunique()
)

# Check condition balance
condition_balance = pd.crosstab(
    validation_variants["variant_name"],
    validation_variants["condition"]
)

row_totals = condition_balance.sum(axis=1)

# For a 45-item form, the four condition counts
# should always be some ordering of 11, 11, 11, 12.
balance_ok = condition_balance.apply(
    lambda row: sorted(row.tolist()) == [11, 11, 11, 12],
    axis=1
)

print("Total forms:", len(form_sizes))

print(
    "Forms with exactly 45 sentences:",
    (form_sizes == 45).sum()
)

print(
    "Forms with exactly 45 unique families:",
    (family_counts == 45).sum()
)

print(
    "Forms with correct 11/11/11/12 condition balance:",
    balance_ok.sum()
)

print(
    "Forms with problems:",
    (~balance_ok).sum()
)

if (~balance_ok).any():
    print("\nPROBLEM FORMS")
    display(
        condition_balance.loc[
            ~balance_ok
        ]
    )
else:
    print("\nAll 16 forms pass the structural and balance checks.")

In [ ]:
# Check that every form contains exactly 45 sentences
form_sizes = (
    validation_variants
    .groupby("variant_name")
    .size()
)

# Check that every form has 45 unique factual families
family_counts = (
    validation_variants
    .groupby("variant_name")["family_id"]
    .nunique()
)

# Check condition balance
condition_balance = pd.crosstab(
    validation_variants["variant_name"],
    validation_variants["condition"]
)

balance_ok = condition_balance.apply(
    lambda row: sorted(row.tolist()) == [11, 11, 11, 12],
    axis=1
)

print("Total forms:", len(form_sizes))

print(
    "Forms with exactly 45 sentences:",
    (form_sizes == 45).sum()
)

print(
    "Forms with exactly 45 unique families:",
    (family_counts == 45).sum()
)

print(
    "Forms with correct 11/11/11/12 condition balance:",
    balance_ok.sum()
)

print(
    "Forms with problems:",
    (~balance_ok).sum()
)

if (~balance_ok).any():
    print("\nPROBLEM FORMS")
    display(
        condition_balance.loc[
            ~balance_ok
        ]
    )
else:
    print(
        "\nAll 16 forms pass the structural and balance checks."
    )

In [ ]:
# Check that every form contains exactly 45 sentences
form_sizes = (
    validation_variants
    .groupby("variant_name")
    .size()
)

# Check that every form has 45 unique factual families
family_counts = (
    validation_variants
    .groupby("variant_name")["family_id"]
    .nunique()
)

# Check condition balance
condition_balance = pd.crosstab(
    validation_variants["variant_name"],
    validation_variants["condition"]
)

balance_ok = condition_balance.apply(
    lambda row: sorted(row.tolist()) == [11, 11, 11, 12],
    axis=1
)

print("Total forms:", len(form_sizes))

print(
    "Forms with exactly 45 sentences:",
    (form_sizes == 45).sum()
)

print(
    "Forms with exactly 45 unique families:",
    (family_counts == 45).sum()
)

print(
    "Forms with correct 11/11/11/12 condition balance:",
    balance_ok.sum()
)

print(
    "Forms with problems:",
    (~balance_ok).sum()
)

if (~balance_ok).any():
    print("\nPROBLEM FORMS")
    display(
        condition_balance.loc[
            ~balance_ok
        ]
    )
else:
    print(
        "\nAll 16 forms pass the structural and balance checks."
    )

In [4]:
print(
    "validation_variants exists:",
    "validation_variants" in globals()
)

if "validation_variants" in globals():
    print(
        "Rows:",
        len(validation_variants)
    )

    print(
        "Variants:",
        validation_variants["variant_name"].nunique()
    )

validation_variants exists: False


In [5]:
import pandas as pd
import numpy as np
import os
import glob

# ============================================================
# 1. Find and load the verified 60-family stimulus bank
# ============================================================

exact_file = "/content/confirmatory_candidate_bank_F001_F060_verified.csv"

if os.path.exists(exact_file):
    BANK_FILE = exact_file
else:
    possible_files = [
        f for f in glob.glob(
            "/content/confirmatory_candidate_bank_F001_F060_verified*.csv"
        )
        if "_audit" not in f
    ]

    if len(possible_files) == 0:
        raise FileNotFoundError(
            "Please upload confirmatory_candidate_bank_F001_F060_verified.csv "
            "to Colab and run this cell again."
        )

    BANK_FILE = possible_files[0]

bank = pd.read_csv(BANK_FILE)

print("Loaded:", BANK_FILE)
print("Families:", len(bank))
print("Unique family IDs:", bank["family_id"].nunique())


# ============================================================
# 2. Recreate the four master counterbalanced lists
# ============================================================

conditions = [
    "true_plain",
    "true_formal",
    "false_plain",
    "false_formal"
]

list_names = [
    "List_A",
    "List_B",
    "List_C",
    "List_D"
]

assignment_rows = []

for family_index, row in bank.reset_index(drop=True).iterrows():

    for list_index, list_name in enumerate(list_names):

        condition = conditions[
            (family_index + list_index) % 4
        ]

        truth, register = condition.split("_")

        assignment_rows.append({
            "list_name": list_name,
            "family_id": row["family_id"],
            "domain": row["domain"],
            "relation": row["relation"],
            "condition": condition,
            "truth": truth,
            "register": register,
            "sentence": row[condition]
        })

sentence_master = pd.DataFrame(assignment_rows)


# ============================================================
# 3. Recreate stable neutral stimulus IDs
# ============================================================

condition_order = {
    "true_plain": 0,
    "true_formal": 1,
    "false_plain": 2,
    "false_formal": 3
}

sentence_master["_condition_order"] = (
    sentence_master["condition"]
    .map(condition_order)
)

sentence_master = (
    sentence_master
    .sort_values(
        ["family_id", "_condition_order"]
    )
    .reset_index(drop=True)
)

sentence_master["stimulus_id"] = [
    f"S{i:03d}"
    for i in range(
        1,
        len(sentence_master) + 1
    )
]

sentence_master = sentence_master.drop(
    columns="_condition_order"
)


# ============================================================
# 4. Recreate the four omission groups
# ============================================================

omission_groups = {
    "G1": [
        "F001", "F005", "F006", "F007", "F011",
        "F012", "F014", "F016", "F020", "F023",
        "F026", "F032", "F043", "F045", "F050"
    ],

    "G2": [
        "F002", "F013", "F015", "F017", "F027",
        "F028", "F029", "F036", "F037", "F042",
        "F046", "F048", "F051", "F055", "F060"
    ],

    "G3": [
        "F008", "F010", "F019", "F021", "F022",
        "F024", "F025", "F030", "F031", "F038",
        "F041", "F044", "F052", "F057", "F059"
    ],

    "G4": [
        "F003", "F004", "F009", "F018", "F033",
        "F034", "F035", "F039", "F040", "F047",
        "F049", "F053", "F054", "F056", "F058"
    ]
}


# ============================================================
# 5. Recreate all sixteen 45-sentence validation variants
# ============================================================

variant_rows = []

for list_index, list_name in enumerate(list_names):

    base_list = sentence_master[
        sentence_master["list_name"] == list_name
    ].copy()

    for group_index, (
        group_name,
        omitted_families
    ) in enumerate(
        omission_groups.items(),
        start=1
    ):

        variant_name = f"{list_name}_V{group_index}"

        variant = base_list[
            ~base_list["family_id"].isin(
                omitted_families
            )
        ].copy()

        random_seed = (
            2026
            + list_index * 100
            + group_index
        )

        variant = (
            variant
            .sample(
                frac=1,
                random_state=random_seed
            )
            .reset_index(drop=True)
        )

        variant["variant_name"] = variant_name

        variant["presentation_order"] = range(
            1,
            len(variant) + 1
        )

        variant["block"] = (
            (
                variant["presentation_order"]
                - 1
            ) // 15
            + 1
        )

        variant_rows.append(variant)

validation_variants = pd.concat(
    variant_rows,
    ignore_index=True
)


# ============================================================
# 6. Run the final structural checks immediately
# ============================================================

form_sizes = (
    validation_variants
    .groupby("variant_name")
    .size()
)

family_counts = (
    validation_variants
    .groupby("variant_name")["family_id"]
    .nunique()
)

condition_balance = pd.crosstab(
    validation_variants["variant_name"],
    validation_variants["condition"]
)

balance_ok = condition_balance.apply(
    lambda row: sorted(row.tolist()) == [11, 11, 11, 12],
    axis=1
)

stimulus_coverage = (
    validation_variants["stimulus_id"]
    .value_counts()
)

print("\nFINAL RECOVERY CHECK")
print("--------------------")

print(
    "validation_variants exists:",
    "validation_variants" in globals()
)

print(
    "Total rows:",
    len(validation_variants)
)

print(
    "Total forms:",
    validation_variants["variant_name"].nunique()
)

print(
    "Forms with exactly 45 sentences:",
    (form_sizes == 45).sum()
)

print(
    "Forms with exactly 45 unique families:",
    (family_counts == 45).sum()
)

print(
    "Forms with correct 11/11/11/12 balance:",
    balance_ok.sum()
)

print(
    "Unique stimuli:",
    stimulus_coverage.size
)

print(
    "Minimum appearances:",
    stimulus_coverage.min()
)

print(
    "Maximum appearances:",
    stimulus_coverage.max()
)

if (
    len(validation_variants) == 720
    and validation_variants["variant_name"].nunique() == 16
    and (form_sizes == 45).all()
    and (family_counts == 45).all()
    and balance_ok.all()
    and stimulus_coverage.size == 240
    and stimulus_coverage.min() == 3
    and stimulus_coverage.max() == 3
):
    print(
        "\nSUCCESS: All 16 validation forms have been restored correctly."
    )
else:
    print(
        "\nWARNING: One or more validation checks did not pass."
    )

FileNotFoundError: Please upload confirmatory_candidate_bank_F001_F060_verified.csv to Colab and run this cell again.

In [6]:
import pandas as pd
import numpy as np
import os
import glob

# ============================================================
# 1. Find and load the verified 60-family stimulus bank
# ============================================================

exact_file = "/content/confirmatory_candidate_bank_F001_F060_verified.csv"

if os.path.exists(exact_file):
    BANK_FILE = exact_file
else:
    possible_files = [
        f for f in glob.glob(
            "/content/confirmatory_candidate_bank_F001_F060_verified*.csv"
        )
        if "_audit" not in f
    ]

    if len(possible_files) == 0:
        raise FileNotFoundError(
            "Please upload confirmatory_candidate_bank_F001_F060_verified.csv "
            "to Colab and run this cell again."
        )

    BANK_FILE = possible_files[0]

bank = pd.read_csv(BANK_FILE)

print("Loaded:", BANK_FILE)
print("Families:", len(bank))
print("Unique family IDs:", bank["family_id"].nunique())


# ============================================================
# 2. Recreate the four master counterbalanced lists
# ============================================================

conditions = [
    "true_plain",
    "true_formal",
    "false_plain",
    "false_formal"
]

list_names = [
    "List_A",
    "List_B",
    "List_C",
    "List_D"
]

assignment_rows = []

for family_index, row in bank.reset_index(drop=True).iterrows():

    for list_index, list_name in enumerate(list_names):

        condition = conditions[
            (family_index + list_index) % 4
        ]

        truth, register = condition.split("_")

        assignment_rows.append({
            "list_name": list_name,
            "family_id": row["family_id"],
            "domain": row["domain"],
            "relation": row["relation"],
            "condition": condition,
            "truth": truth,
            "register": register,
            "sentence": row[condition]
        })

sentence_master = pd.DataFrame(assignment_rows)


# ============================================================
# 3. Recreate stable neutral stimulus IDs
# ============================================================

condition_order = {
    "true_plain": 0,
    "true_formal": 1,
    "false_plain": 2,
    "false_formal": 3
}

sentence_master["_condition_order"] = (
    sentence_master["condition"]
    .map(condition_order)
)

sentence_master = (
    sentence_master
    .sort_values(
        ["family_id", "_condition_order"]
    )
    .reset_index(drop=True)
)

sentence_master["stimulus_id"] = [
    f"S{i:03d}"
    for i in range(
        1,
        len(sentence_master) + 1
    )
]

sentence_master = sentence_master.drop(
    columns="_condition_order"
)


# ============================================================
# 4. Recreate the four omission groups
# ============================================================

omission_groups = {
    "G1": [
        "F001", "F005", "F006", "F007", "F011",
        "F012", "F014", "F016", "F020", "F023",
        "F026", "F032", "F043", "F045", "F050"
    ],

    "G2": [
        "F002", "F013", "F015", "F017", "F027",
        "F028", "F029", "F036", "F037", "F042",
        "F046", "F048", "F051", "F055", "F060"
    ],

    "G3": [
        "F008", "F010", "F019", "F021", "F022",
        "F024", "F025", "F030", "F031", "F038",
        "F041", "F044", "F052", "F057", "F059"
    ],

    "G4": [
        "F003", "F004", "F009", "F018", "F033",
        "F034", "F035", "F039", "F040", "F047",
        "F049", "F053", "F054", "F056", "F058"
    ]
}


# ============================================================
# 5. Recreate all sixteen 45-sentence validation variants
# ============================================================

variant_rows = []

for list_index, list_name in enumerate(list_names):

    base_list = sentence_master[
        sentence_master["list_name"] == list_name
    ].copy()

    for group_index, (
        group_name,
        omitted_families
    ) in enumerate(
        omission_groups.items(),
        start=1
    ):

        variant_name = f"{list_name}_V{group_index}"

        variant = base_list[
            ~base_list["family_id"].isin(
                omitted_families
            )
        ].copy()

        random_seed = (
            2026
            + list_index * 100
            + group_index
        )

        variant = (
            variant
            .sample(
                frac=1,
                random_state=random_seed
            )
            .reset_index(drop=True)
        )

        variant["variant_name"] = variant_name

        variant["presentation_order"] = range(
            1,
            len(variant) + 1
        )

        variant["block"] = (
            (
                variant["presentation_order"]
                - 1
            ) // 15
            + 1
        )

        variant_rows.append(variant)

validation_variants = pd.concat(
    variant_rows,
    ignore_index=True
)


# ============================================================
# 6. Run the final structural checks immediately
# ============================================================

form_sizes = (
    validation_variants
    .groupby("variant_name")
    .size()
)

family_counts = (
    validation_variants
    .groupby("variant_name")["family_id"]
    .nunique()
)

condition_balance = pd.crosstab(
    validation_variants["variant_name"],
    validation_variants["condition"]
)

balance_ok = condition_balance.apply(
    lambda row: sorted(row.tolist()) == [11, 11, 11, 12],
    axis=1
)

stimulus_coverage = (
    validation_variants["stimulus_id"]
    .value_counts()
)

print("\nFINAL RECOVERY CHECK")
print("--------------------")

print(
    "validation_variants exists:",
    "validation_variants" in globals()
)

print(
    "Total rows:",
    len(validation_variants)
)

print(
    "Total forms:",
    validation_variants["variant_name"].nunique()
)

print(
    "Forms with exactly 45 sentences:",
    (form_sizes == 45).sum()
)

print(
    "Forms with exactly 45 unique families:",
    (family_counts == 45).sum()
)

print(
    "Forms with correct 11/11/11/12 balance:",
    balance_ok.sum()
)

print(
    "Unique stimuli:",
    stimulus_coverage.size
)

print(
    "Minimum appearances:",
    stimulus_coverage.min()
)

print(
    "Maximum appearances:",
    stimulus_coverage.max()
)

if (
    len(validation_variants) == 720
    and validation_variants["variant_name"].nunique() == 16
    and (form_sizes == 45).all()
    and (family_counts == 45).all()
    and balance_ok.all()
    and stimulus_coverage.size == 240
    and stimulus_coverage.min() == 3
    and stimulus_coverage.max() == 3
):
    print(
        "\nSUCCESS: All 16 validation forms have been restored correctly."
    )
else:
    print(
        "\nWARNING: One or more validation checks did not pass."
    )

FileNotFoundError: Please upload confirmatory_candidate_bank_F001_F060_verified.csv to Colab and run this cell again.

In [7]:
from google.colab import files

uploaded = files.upload()

Saving confirmatory_candidate_bank_F001_F060_verified.csv to confirmatory_candidate_bank_F001_F060_verified.csv


In [8]:
import os

print(
    os.path.exists(
        "/content/confirmatory_candidate_bank_F001_F060_verified.csv"
    )
)

True


In [9]:
print(
    "validation_variants exists:",
    "validation_variants" in globals()
)

if "validation_variants" in globals():
    print(
        "Rows:",
        len(validation_variants)
    )

    print(
        "Variants:",
        validation_variants["variant_name"].nunique()
    )

validation_variants exists: False


In [10]:
import pandas as pd
import numpy as np
import os
import glob

# ============================================================
# 1. Find and load the verified 60-family stimulus bank
# ============================================================

exact_file = "/content/confirmatory_candidate_bank_F001_F060_verified.csv"

if os.path.exists(exact_file):
    BANK_FILE = exact_file
else:
    possible_files = [
        f for f in glob.glob(
            "/content/confirmatory_candidate_bank_F001_F060_verified*.csv"
        )
        if "_audit" not in f
    ]

    if len(possible_files) == 0:
        raise FileNotFoundError(
            "Please upload confirmatory_candidate_bank_F001_F060_verified.csv "
            "to Colab and run this cell again."
        )

    BANK_FILE = possible_files[0]

bank = pd.read_csv(BANK_FILE)

print("Loaded:", BANK_FILE)
print("Families:", len(bank))
print("Unique family IDs:", bank["family_id"].nunique())


# ============================================================
# 2. Recreate the four master counterbalanced lists
# ============================================================

conditions = [
    "true_plain",
    "true_formal",
    "false_plain",
    "false_formal"
]

list_names = [
    "List_A",
    "List_B",
    "List_C",
    "List_D"
]

assignment_rows = []

for family_index, row in bank.reset_index(drop=True).iterrows():

    for list_index, list_name in enumerate(list_names):

        condition = conditions[
            (family_index + list_index) % 4
        ]

        truth, register = condition.split("_")

        assignment_rows.append({
            "list_name": list_name,
            "family_id": row["family_id"],
            "domain": row["domain"],
            "relation": row["relation"],
            "condition": condition,
            "truth": truth,
            "register": register,
            "sentence": row[condition]
        })

sentence_master = pd.DataFrame(assignment_rows)


# ============================================================
# 3. Recreate stable neutral stimulus IDs
# ============================================================

condition_order = {
    "true_plain": 0,
    "true_formal": 1,
    "false_plain": 2,
    "false_formal": 3
}

sentence_master["_condition_order"] = (
    sentence_master["condition"]
    .map(condition_order)
)

sentence_master = (
    sentence_master
    .sort_values(
        ["family_id", "_condition_order"]
    )
    .reset_index(drop=True)
)

sentence_master["stimulus_id"] = [
    f"S{i:03d}"
    for i in range(
        1,
        len(sentence_master) + 1
    )
]

sentence_master = sentence_master.drop(
    columns="_condition_order"
)


# ============================================================
# 4. Recreate the four omission groups
# ============================================================

omission_groups = {
    "G1": [
        "F001", "F005", "F006", "F007", "F011",
        "F012", "F014", "F016", "F020", "F023",
        "F026", "F032", "F043", "F045", "F050"
    ],

    "G2": [
        "F002", "F013", "F015", "F017", "F027",
        "F028", "F029", "F036", "F037", "F042",
        "F046", "F048", "F051", "F055", "F060"
    ],

    "G3": [
        "F008", "F010", "F019", "F021", "F022",
        "F024", "F025", "F030", "F031", "F038",
        "F041", "F044", "F052", "F057", "F059"
    ],

    "G4": [
        "F003", "F004", "F009", "F018", "F033",
        "F034", "F035", "F039", "F040", "F047",
        "F049", "F053", "F054", "F056", "F058"
    ]
}


# ============================================================
# 5. Recreate all sixteen 45-sentence validation variants
# ============================================================

variant_rows = []

for list_index, list_name in enumerate(list_names):

    base_list = sentence_master[
        sentence_master["list_name"] == list_name
    ].copy()

    for group_index, (
        group_name,
        omitted_families
    ) in enumerate(
        omission_groups.items(),
        start=1
    ):

        variant_name = f"{list_name}_V{group_index}"

        variant = base_list[
            ~base_list["family_id"].isin(
                omitted_families
            )
        ].copy()

        random_seed = (
            2026
            + list_index * 100
            + group_index
        )

        variant = (
            variant
            .sample(
                frac=1,
                random_state=random_seed
            )
            .reset_index(drop=True)
        )

        variant["variant_name"] = variant_name

        variant["presentation_order"] = range(
            1,
            len(variant) + 1
        )

        variant["block"] = (
            (
                variant["presentation_order"]
                - 1
            ) // 15
            + 1
        )

        variant_rows.append(variant)

validation_variants = pd.concat(
    variant_rows,
    ignore_index=True
)


# ============================================================
# 6. Run the final structural checks immediately
# ============================================================

form_sizes = (
    validation_variants
    .groupby("variant_name")
    .size()
)

family_counts = (
    validation_variants
    .groupby("variant_name")["family_id"]
    .nunique()
)

condition_balance = pd.crosstab(
    validation_variants["variant_name"],
    validation_variants["condition"]
)

balance_ok = condition_balance.apply(
    lambda row: sorted(row.tolist()) == [11, 11, 11, 12],
    axis=1
)

stimulus_coverage = (
    validation_variants["stimulus_id"]
    .value_counts()
)

print("\nFINAL RECOVERY CHECK")
print("--------------------")

print(
    "validation_variants exists:",
    "validation_variants" in globals()
)

print(
    "Total rows:",
    len(validation_variants)
)

print(
    "Total forms:",
    validation_variants["variant_name"].nunique()
)

print(
    "Forms with exactly 45 sentences:",
    (form_sizes == 45).sum()
)

print(
    "Forms with exactly 45 unique families:",
    (family_counts == 45).sum()
)

print(
    "Forms with correct 11/11/11/12 balance:",
    balance_ok.sum()
)

print(
    "Unique stimuli:",
    stimulus_coverage.size
)

print(
    "Minimum appearances:",
    stimulus_coverage.min()
)

print(
    "Maximum appearances:",
    stimulus_coverage.max()
)

if (
    len(validation_variants) == 720
    and validation_variants["variant_name"].nunique() == 16
    and (form_sizes == 45).all()
    and (family_counts == 45).all()
    and balance_ok.all()
    and stimulus_coverage.size == 240
    and stimulus_coverage.min() == 3
    and stimulus_coverage.max() == 3
):
    print(
        "\nSUCCESS: All 16 validation forms have been restored correctly."
    )
else:
    print(
        "\nWARNING: One or more validation checks did not pass."
    )

Loaded: /content/confirmatory_candidate_bank_F001_F060_verified.csv
Families: 60
Unique family IDs: 60

FINAL RECOVERY CHECK
--------------------
validation_variants exists: True
Total rows: 720
Total forms: 16
Forms with exactly 45 sentences: 16
Forms with exactly 45 unique families: 16
Forms with correct 11/11/11/12 balance: 16
Unique stimuli: 240
Minimum appearances: 3
Maximum appearances: 3

SUCCESS: All 16 validation forms have been restored correctly.


In [11]:
import os

OUTPUT_DIR = "/content/confirmatory_validation_rater_sheets"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

rating_columns = [
    "formality_1_7",
    "assertiveness_confidence_1_7",
    "politeness_1_7",
    "naturalness_1_7",
    "expertise_authority_1_7",
    "evidential_strength_1_7"
]

created_files = []

for variant_name, group in validation_variants.groupby(
    "variant_name"
):

    rater_sheet = (
        group[
            [
                "presentation_order",
                "block",
                "stimulus_id",
                "sentence"
            ]
        ]
        .sort_values("presentation_order")
        .reset_index(drop=True)
        .copy()
    )

    # Add blank rating columns
    for column in rating_columns:
        rater_sheet[column] = ""

    filename = (
        f"{variant_name}_rater_sheet.csv"
    )

    filepath = os.path.join(
        OUTPUT_DIR,
        filename
    )

    rater_sheet.to_csv(
        filepath,
        index=False
    )

    created_files.append(filepath)

print(
    "Rater sheets created:",
    len(created_files)
)

print(
    "Output folder:",
    OUTPUT_DIR
)

Rater sheets created: 16
Output folder: /content/confirmatory_validation_rater_sheets


In [12]:
example_file = os.path.join(
    OUTPUT_DIR,
    "List_A_V1_rater_sheet.csv"
)

example_sheet = pd.read_csv(
    example_file
)

print(
    "Rows:",
    len(example_sheet)
)

display(
    example_sheet.head(10)
)

Rows: 45


,presentation_order,block,stimulus_id,sentence,formality_1_7,assertiveness_confidence_1_7,politeness_1_7,naturalness_1_7,expertise_authority_1_7,evidential_strength_1_7
0,1,1,S240,I think Henry Moore created the sculpture The ...,NaN,NaN,NaN,NaN,NaN,NaN
1,2,1,S049,I think the heart pumps blood through the body.,NaN,NaN,NaN,NaN,NaN,NaN
2,3,1,S011,I think Venus is the closest planet to the Sun.,NaN,NaN,NaN,NaN,NaN,NaN
3,4,1,S235,I think German verbs are capitalized even when...,NaN,NaN,NaN,NaN,NaN,NaN
4,5,1,S038,I think Neil Armstrong was the first human to ...,NaN,NaN,NaN,NaN,NaN,NaN
5,6,1,S075,I think the Sahara Desert is in Asia.,NaN,NaN,NaN,NaN,NaN,NaN
6,7,1,S208,I think the joule constitutes the SI unit of f...,NaN,NaN,NaN,NaN,NaN,NaN
7,8,1,S112,I think Mars possesses three natural satellites.,NaN,NaN,NaN,NaN,NaN,NaN
8,9,1,S134,I think the American Declaration of Independen...,NaN,NaN,NaN,NaN,NaN,NaN
9,10,1,S193,I think Earth's inner core is in solid form.,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
expected_columns = [
    "presentation_order",
    "block",
    "stimulus_id",
    "sentence",
    "formality_1_7",
    "assertiveness_confidence_1_7",
    "politeness_1_7",
    "naturalness_1_7",
    "expertise_authority_1_7",
    "evidential_strength_1_7"
]

problems = []

for filepath in created_files:

    sheet = pd.read_csv(filepath)

    if len(sheet) != 45:
        problems.append(
            (
                filepath,
                "wrong number of rows"
            )
        )

    if list(sheet.columns) != expected_columns:
        problems.append(
            (
                filepath,
                "wrong columns"
            )
        )

    if sheet["stimulus_id"].nunique() != 45:
        problems.append(
            (
                filepath,
                "duplicate stimulus IDs"
            )
        )

    if sorted(
        sheet["block"].unique().tolist()
    ) != [1, 2, 3]:
        problems.append(
            (
                filepath,
                "block problem"
            )
        )

print(
    "Number of sheet problems:",
    len(problems)
)

if problems:
    display(
        pd.DataFrame(
            problems,
            columns=[
                "file",
                "problem"
            ]
        )
    )
else:
    print(
        "SUCCESS: All 16 blinded rater sheets are structurally correct."
    )

Number of sheet problems: 0
SUCCESS: All 16 blinded rater sheets are structurally correct.


In [14]:
hidden_master_path = (
    "/content/"
    "confirmatory_validation_hidden_master.csv"
)

validation_variants.to_csv(
    hidden_master_path,
    index=False
)

print(
    "Hidden master saved:",
    hidden_master_path
)

Hidden master saved: /content/confirmatory_validation_hidden_master.csv


In [15]:
import zipfile

zip_path = (
    "/content/"
    "confirmatory_validation_16_rater_sheets.zip"
)

with zipfile.ZipFile(
    zip_path,
    "w"
) as zip_file:

    for filepath in created_files:

        zip_file.write(
            filepath,
            arcname=os.path.basename(
                filepath
            )
        )

print(
    "ZIP created:",
    zip_path
)

ZIP created: /content/confirmatory_validation_16_rater_sheets.zip


In [16]:
from google.colab import files

files.download(
    "/content/"
    "confirmatory_validation_16_rater_sheets.zip"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
files.download(
    "/content/"
    "confirmatory_validation_hidden_master.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>